In this assignment, we'll implement a sequence-to-sequence Transformer model for an English-to-Spanish machine translation task. The goals are as follow:

- familiarize yourself with preparing text data and vectorizing text using the Keras TextVectorization layer.  
- Implement sinusoidal positional encoding using sine and cosine functions as presented in the lecture and original Transformer paper (i.e., 'Attention Is All You Need').
- Implement multi-head attention machenism from scratch.
- Run Transformer encoder and Transformer decoder to train the sequence-to-sequence model.
- Use the trained model to generate translations from the input sentences

Note that this assignment should be implemented using the Keras version 2.

In [1]:
import pathlib
import random
import string
import re
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
import tensorflow.data as tf_data
import tensorflow.strings as tf_strings

import keras
from keras import layers
from keras.layers import TextVectorization

2026-03-01 14:34:44.655901: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-01 14:34:44.657672: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-01 14:34:44.687650: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-01 14:34:44.687669: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-01 14:34:44.688521: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

# Preparing text data for machine translation


When constructing a neural machine translation (NMT) system, it's necessary to acquire data for both training and testing. Suppose we aim to develop an English-to-Spanish translator based on sentences. Numerous online resources are available for this purpose, such as user-generated content found in apps like Anki, specifically designed for language learning. Typically, this data is packaged within a ZIP file, encapsulating a SQLite database. From this repository, English-Spanish sentence pairs can be extracted for the translation model.

In [2]:
text_file = keras.utils.get_file(
    fname="spa-eng.zip",
    origin="http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip",
    extract=True,
)
text_file = pathlib.Path(text_file).parent / "spa-eng" / "spa.txt"

Each line contains an English sentence and its corresponding Spanish sentence. The English sentence is the source sequence and Spanish one is the target sequence. We prepend the token "[start]" and we append the token "[end]" to the Spanish sentence.

In [3]:
with open(text_file) as f:
    lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
    eng, spa = line.split("\t")
    spa = "[start] " + spa + " [end]"
    text_pairs.append((eng, spa))

Here's what our sentence pairs look like:

In [4]:
for _ in range(5):
    print(random.choice(text_pairs))

('Enjoy your meal.', '[start] Buen provecho. [end]')
("I'll understand.", '[start] Lo entenderé. [end]')
('Thank you for inviting us to dinner.', '[start] Gracias por invitarnos a cenar. [end]')
("You didn't listen.", '[start] Vos no escuchaste. [end]')
('She stayed at the hotel for several days.', '[start] Ella se quedó en el hotel varios días. [end]')


Now, let's split the sentence pairs into a training set, a validation set, and a test set.

In [5]:
random.shuffle(text_pairs)
num_val_samples = int(0.15 * len(text_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples
train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples : num_train_samples + num_val_samples]
test_pairs = text_pairs[num_train_samples + num_val_samples :]

print(f"{len(text_pairs)} total pairs")
print(f"{len(train_pairs)} training pairs")
print(f"{len(val_pairs)} validation pairs")
print(f"{len(test_pairs)} test pairs")

118964 total pairs
83276 training pairs
17844 validation pairs
17844 test pairs


# Vectorizing the text data

We'll use two instances of the TextVectorization layer to vectorize the text data (one for English and one for Spanish), that is to say, to turn the original strings into integer sequences where each integer represents the index of a word in a vocabulary.

The English layer will use the default string standardization (strip punctuation characters) and splitting scheme (split on whitespace), while the Spanish layer will use a custom standardization, where we add the character "¿" to the set of punctuation characters to be stripped.

Note: in a production-grade machine translation model, I would not recommend stripping the punctuation characters in either language. Instead, I would recommend turning each punctuation character into its own token, which you could achieve by providing a custom split function to the TextVectorization layer.

In [6]:
strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

vocab_size = 15000
sequence_length = 20
batch_size = 64


def custom_standardization(input_string):
    lowercase = tf_strings.lower(input_string)
    return tf_strings.regex_replace(lowercase, "[%s]" % re.escape(strip_chars), "")


eng_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)
spa_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)
train_eng_texts = [pair[0] for pair in train_pairs]
train_spa_texts = [pair[1] for pair in train_pairs]
eng_vectorization.adapt(train_eng_texts)
spa_vectorization.adapt(train_spa_texts)

2026-03-01 14:34:58.807509: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Next, we'll format our datasets.

At each training step, the model will seek to predict target words N+1 (and beyond) using the source sentence and the target words 0 to N.

As such, the training dataset will yield a tuple (inputs, targets), where the input is a dict with keys `encoder_inputs` and `decoder_inputs`, each is a vector, corresponding to English and Spanish sentences respectively. The target is also vector of the Spanish sentence, advanced by 1 token. All vector are in the same length.

The output will be used for training the transformer model. In the model we will create, the input tensors are named `encoder_inputs` and `decoder_inputs` which should be matched to the keys in the dictionary for the source part.

In [7]:
def format_dataset(eng, spa):
    eng = eng_vectorization(eng)
    spa = spa_vectorization(spa)
    return (
        {
            "encoder_inputs": eng,
            "decoder_inputs": spa[:, :-1],
        },
        spa[:, 1:],
    )


def make_dataset(pairs):
    eng_texts, spa_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    spa_texts = list(spa_texts)
    dataset = tf_data.Dataset.from_tensor_slices((eng_texts, spa_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset)
    return dataset.cache().shuffle(2048).prefetch(16)


train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

Let's take a quick look at the sequence shapes (we have batches of 64 pairs, and all sequences are 20 steps long). It means that We assumed that a sentence should have no more than 20 tokens.

In [8]:
for inputs, targets in train_ds.take(1):
    print(f'inputs["encoder_inputs"].shape: {inputs["encoder_inputs"].shape}')
    print(f'inputs["encoder_inputs"][0]: {inputs["encoder_inputs"][0]}')
    print(f'inputs["decoder_inputs"].shape: {inputs["decoder_inputs"].shape}')
    print(f'inputs["decoder_inputs"][0]: {inputs["decoder_inputs"][0]}')
    print(f"targets.shape: {targets.shape}")
    print(f"targets[0]: {targets[0]}")

inputs["encoder_inputs"].shape: (64, 20)
inputs["encoder_inputs"][0]: [ 18   8  24   9 115   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0]
inputs["decoder_inputs"].shape: (64, 20)
inputs["decoder_inputs"][0]: [ 2 56 12 17  5 92  3  0  0  0  0  0  0  0  0  0  0  0  0  0]
targets.shape: (64, 20)
targets[0]: [56 12 17  5 92  3  0  0  0  0  0  0  0  0  0  0  0  0  0  0]


In [9]:
print(inputs["encoder_inputs"].shape)

(64, 20)


# **Task 1: Positional encoding**

Upon vectorizing a sentence, we get a resulting vector of integers where each integer denoting a specific word. However, it's important to note that these integers merely serve as labels; proximity between two integers doesn't imply semantic relation between their corresponding words.

To understand word meanings and establish relational measures between them, one employs word embeddings. But to understand the context, you also need to know the position of each word in a sentence. This is done by positional encoding.

In this task, you need to implement sinusoidal positional encoding using sine and cosine functions as presented in the lecture. Specifically, in the original Transformer paper,the author used a sinusoidal function to get positional encoding which will append to encoder and decoder word embeddings. This is to give information about the position of each token. The function looks as follows:

 $$\Large{PE_{(k, 2i)} = \sin(pos / 10000^{2i / d_{model}})} $$
$$\Large{PE_{(k, 2i+1)} = \cos(pos / 10000^{2i / d_{model}})} $$

Where, $k = 0, 1, .., L-1$ ($L$ is the length of input sequence), and $i=0, 1, .., d/2$.

You can write your own position encoding layer, using model subclassing inherited from the base class ('layers.Layer') from Keras. Complete the code below with the basic structure of the custom layer class:

In [ ]:
class SinusoidalPositionalEncoding(layers.Layer):
  def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
    super().__init__(**kwargs)
    
    self.sequence_length = sequence_length
    self.vocab_size = vocab_size
    self.embed_dim = embed_dim  

    self.get_embeddings = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)

    pos_index = np.arange(self.sequence_length).reshape(self.sequence_length, 1)  # (sequence_length=20, 1)

    denom = np.power(10000, (2 * (np.arange(self.embed_dim) // 2)) / self.embed_dim)  # (embed_dim=,)
    angles = pos_index / denom  

    angles[:, 0::2] = np.sin(angles[:, 0::2]) # even index xvalues 
    angles[:, 1::2] = np.cos(angles[:, 1::2])

    self.positional_embeddings = tf.constant(angles, dtype=tf.float32)[None, :, :]  # (1, sequence_length, embed_dim)
    
  
  def call(self, inputs):
    
    embedding_vectors = self.get_embeddings(inputs)  # (batch_size, sequence_length, embed_dim)
    new_embeddings = embedding_vectors + self.positional_embeddings  

    return new_embeddings

Here, input is the tokenized sentence (refer to an example of "encoder input" above). The positional embedding should return a tensor with shape $B×L×d_{model}$. You can add more subclasses or functions with suitable arguments if needed.

# **Task 2: Multi-head attention**

Each multi-head attention block contains four steps:

- Three linear (dense) layers that each receive the queries, keys, or values
- Scaled dot-product attention operations performed h times in parallel, according to the number of heads composing the multi-head attention block.
- Concatenate the outputs of the different heads
- A final linear (dense) layer that produces the output

In this task, you need to implement multi-head attention machenism from scratch. Specifically, you need to complete the code below for 'ScaledDotProductAttention' and 'MultiHeadAttentionCustom' classes, which inherit from the Layer base class in Keras:



In [ ]:
class ScaledDotProductAttention(layers.Layer):
  def __init__(self):
    super().__init__()

  def call(self, query, key, value, mask=None):
    query_k = tf.matmul(query, key, transpose_b=True)  # QK^T   
    dim = tf.cast(tf.shape(key)[-1])
    attention_logits = query_k / tf.math.sqrt(dim) 
    attention_logits += (mask * 1e-10) if mask is not None else 0  # applied mask also avoiding -inf value in softmax 


    attention_w = tf.nn.softmax(attention_logits, axis=-1)
    output = tf.matmul(attention_w, value)

    return ouput, attention_w 

class MultiHeadAttentionCustom(layers.Layer):
  def __init__(self, num_heds, key_dim):
    super().__init__()
    
    self.n_heads = n_heads
    self.key_dim = key_dim # overall dimension for key_dim * n_heads

    self.query_weights = layers.Dense(key_dim) # same dimensions for qkv
    self.value_weights = layers.Dense(key_dim)
    self.key_weights = layers.Dense(key_dim)

    self.attention = ScaledDotProductAttention()
    self.output_weights = layers.Dense(key_dim)



  def call(self, query, key, value, causal_mask):
    query = self.query_weights(query) # (batch_size, sequence_len, key_dim * n_heads)
    key = self.key_weights(key)
    value = self.value_weights(value)

    batch_size = tf.shape(query)[0]
    query = tf.reshape(query, (batch_size, -1, self.n_heads, self.key_dim // 2))
    key = tf.reshape(key, (batch_size, -1, self.n_heads, self.key_dim // 2))
    value = tf.reshape(value, (batch_size, -1, self.n_heads, self.key_dim // 2))

    query = tf.transpose(query, perm=[0, 2, 1, 3]) # (batch_size, n_heads, sequence_len, key_dim)
    key = tf.transpose(key, perm=[0, 2, 1, 3])
    value = tf.transpose(value, perm=[0, 2, 1, 3])

    # make a mask
    if causal_mask == True:
      # upper diagonal matrix, with 0 for valid positions and 1 for masked positions
      causal_mask = 1 - tf.linalg.band_part(tf.ones((tf.shape(query)[2], tf.shape(key)[2])), -1, 0)  # upper diagonal amtrix, (sequence_len, sequence_len)
      causal_mask = causal_mask[None, None, :, :]  # (1, 1, sequence_len, sequence_len) for broadcasting

    
    attention_output, attention_weights = self.attention(query, key, value, mask=mask)

    attention_output = tf.transpose(attention_output, perm=[0, 2, 1, 3]) # (batch_size, sequence_len, n_heads, key_dim) for concat
    concatenated_attention = tf.reshape(attention_output, (batch_size, -1, self.key_dim) ) # merge heads back together
    output = self.output_weights(concatenated_attention)

    return output




You can add more subclasses or functions with suitable arguments if needed. Note that, for MultiHeadAttentionCustom, you should have a call argument ('causal_mask'), which is a boolean to indicate whether to apply a causal mask to prevent tokens from attending to future tokens (used in a decoder Transformer).

# Encoder and decoder

The flowchart for the whole Transformer architecture is shown below, where the encoder is the block to the left, and decoder is the block to the right. Note that since the output shape of either encoder or decoder is the same as its corresponding input shape, both the encoder unit and the decoder unit can be stacked.
![Transformer Architecture](https://www.tensorflow.org/images/tutorials/transformer/transformer.png)

We then present the transformer encoder architecture. Each encoder has a multihead self-attention (encoder-encoder) sublayer and a feedforward sublayer (two dense layers with ReLU activation in between). Each sublayer is followed by a LayerNorm taking the sublayer residually as follows:

$$\Large{\mathit{LayerNorm}(x + \mathit{sublayer}(x))} $$

In [ ]:
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = MultiHeadAttentionCustom(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(dense_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()


    def call(self, inputs, mask=None):

        attention_output = self.attention(
            query=inputs, value=inputs, key=inputs, causal_mask= False
        )
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "dense_dim": self.dense_dim,
                "num_heads": self.num_heads,
            }
        )
        return config

The decoder follows a similar construction but with three sublayers instead of two: a multihead self-attention layer (decoder-decoder), a multihead encoder attention layer (decoder-encoder), and a feedforward layer similar to the one in an encoder unit.

In [ ]:
class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, latent_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.latent_dim = latent_dim
        self.num_heads = num_heads
        self.attention_1 = MultiHeadAttentionCustom(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.attention_2 = MultiHeadAttentionCustom(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(latent_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, encoder_outputs, mask=None):

        attention_output_1 = self.attention_1(
            query=inputs, value=inputs, key=inputs, causal_mask= True
        )
        out_1 = self.layernorm_1(inputs + attention_output_1)

        attention_output_2 = self.attention_2(
            query=out_1,
            value=encoder_outputs,
            key=encoder_outputs,
            causal_mask= False,
        )
        out_2 = self.layernorm_2(out_1 + attention_output_2)

        proj_output = self.dense_proj(out_2)
        return self.layernorm_3(out_2 + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "latent_dim": self.latent_dim,
                "num_heads": self.num_heads,
            }
        )
        return config

# Assembling the Full Architecture

Now we can build the full architecture of transformer using positional encoding, encoder layers and decoder layers:

In [ ]:
embed_dim = 512
latent_dim = 64
num_heads = 8

encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="encoder_inputs")
x = SinusoidalPositionalEncoding(sequence_length, vocab_size, embed_dim)(encoder_inputs)
encoder_outputs = TransformerEncoder(embed_dim, latent_dim, num_heads)(x)
encoder = keras.Model(encoder_inputs, encoder_outputs)

decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="decoder_inputs")
encoded_seq_inputs = keras.Input(shape=(None, embed_dim), name="decoder_state_inputs")
x = SinusoidalPositionalEncoding(sequence_length, vocab_size, embed_dim)(decoder_inputs)
x = TransformerDecoder(embed_dim, latent_dim, num_heads)(x, encoded_seq_inputs)
x = layers.Dropout(0.5)(x)
decoder_outputs = layers.Dense(vocab_size, activation="softmax")(x)
decoder = keras.Model([decoder_inputs, encoded_seq_inputs], decoder_outputs)

decoder_outputs = decoder([decoder_inputs, encoder_outputs])
transformer = keras.Model(
    [encoder_inputs, decoder_inputs], decoder_outputs, name="transformer"
)

# Train the machine translation model

We'll use accuracy as a quick way to monitor training progress on the validation data. Note that machine translation typically uses BLEU scores as well as other metrics, rather than accuracy.

Here we only train for 1 epoch, but to get the model to actually converge you should train for more epochs.

In [ ]:
epochs = 1

transformer.summary()
transformer.compile(
    "rmsprop", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
transformer.fit(train_ds, epochs=epochs, validation_data=val_ds)

# Translation testing

Finally, let's demonstrate how to translate brand new English sentences. We simply feed into the model the vectorized English sentence as well as the target token "[start]", then we repeatedly generated the next token, until we hit the token "[end]".

In [ ]:
spa_vocab = spa_vectorization.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))
max_decoded_sentence_length = 20


def decode_sequence(input_sentence):
    tokenized_input_sentence = eng_vectorization([input_sentence])
    decoded_sentence = "[start]"
    for i in range(max_decoded_sentence_length):
        tokenized_target_sentence = spa_vectorization([decoded_sentence])[:, :-1]
        predictions = transformer([tokenized_input_sentence, tokenized_target_sentence])

        sampled_token_index = tf.math.argmax(predictions[0, i, :]).numpy().item(0)
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token

        if sampled_token == "[end]":
            break
    return decoded_sentence


test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(30):
    input_sentence = random.choice(test_eng_texts)
    translated = decode_sequence(input_sentence)

# Reference
https://keras.io/examples/nlp/neural_machine_translation_with_transformer/